# HumanEval problem-level stability analysis

This notebook loads the saved **per-run task result CSV files** in the same way as the pass@k notebook, but focuses only on **problem-level pass/fail stability** across models and runs.

It does **not** compute pass@k and does **not** run statistical significance tests.

The goal is to answer questions such as:

- Which HumanEval problems are consistently solved or consistently failed by each model?
- Which problems are unstable across runs?
- Which problems distinguish one model from another?
- Are model gains broad across many problems or concentrated in a few tasks?

In [ ]:
# =========================
# 1. Setup
# =========================
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
pd.set_option('display.max_columns', 100)

In [ ]:
# =========================
# 2. Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# 3. Paths
# =========================
PROJECT_DIR = Path('/content/drive/MyDrive/Speciale/4. Model Evaluation')
RESULTS_DIR = PROJECT_DIR / 'results'
TASK_RESULTS_DIR = RESULTS_DIR / 'per_run_task_results'
TASK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = RESULTS_DIR / 'problem_level_stability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_RUN_COUNTS_CSV = OUTPUT_DIR / 'model_run_counts.csv'
RUN_LEVEL_TASK_MATRIX_CSV = OUTPUT_DIR / 'run_level_task_matrix.csv'
PROBLEM_STABILITY_CSV = OUTPUT_DIR / 'problem_stability_by_model.csv'
PROBLEM_STABILITY_WIDE_CSV = OUTPUT_DIR / 'problem_stability_wide.csv'
PROBLEM_OVERALL_DIFFICULTY_CSV = OUTPUT_DIR / 'problem_overall_difficulty.csv'
MODEL_STABILITY_SUMMARY_CSV = OUTPUT_DIR / 'model_stability_summary.csv'
PAIRWISE_PROBLEM_COMPARISON_CSV = OUTPUT_DIR / 'pairwise_problem_comparison.csv'
UNIQUE_SOLVES_CSV = OUTPUT_DIR / 'unique_and_near_unique_solves.csv'

HEATMAP_PNG = OUTPUT_DIR / 'problem_pass_rate_heatmap.png'
VOLATILITY_PNG = OUTPUT_DIR / 'problem_volatility_by_model.png'
DIFFICULTY_PNG = OUTPUT_DIR / 'problem_difficulty_distribution.png'
STABILITY_BUCKET_PNG = OUTPUT_DIR / 'stability_bucket_counts.png'
PAIRWISE_DELTA_HEATMAP_PNG = OUTPUT_DIR / 'pairwise_mean_pass_rate_delta_heatmap.png'

EXPECTED_RUNS = 10
REFERENCE_MODEL_LABEL = None  # optional: set to a model label to prioritize it in comparisons

print('TASK_RESULTS_DIR:', TASK_RESULTS_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Assumptions

This notebook assumes:

- each CSV corresponds to **one run for one model**
- filenames follow `model_label__run_<n>_task_results.csv`, or contain `model_label` and `run_idx` columns
- `passed` can be interpreted as boolean
- each run contains at most one row per `task_id`
- problem-level stability should be calculated from the observed per-run pass/fail outcomes

In [ ]:
# =========================
# 4. Helper functions for loading saved runs
# =========================
RUN_FILE_PATTERN = re.compile(r'^(?P<model_label>.+)__run_(?P<run_idx>\d+)_task_results\.csv$')

def parse_bool_like(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, np.integer, float, np.floating)):
        return bool(x)
    s = str(x).strip().lower()
    true_values = {'true', '1', 'yes', 'y', 'passed', 'pass'}
    false_values = {'false', '0', 'no', 'n', 'failed', 'fail', ''}
    if s in true_values:
        return True
    if s in false_values:
        return False
    return bool(s)

def discover_task_result_files(task_results_dir: Path):
    files = sorted(task_results_dir.glob('*_task_results.csv'))
    rows = []
    for path in files:
        m = RUN_FILE_PATTERN.match(path.name)
        if m:
            inferred_model = m.group('model_label')
            inferred_run_idx = int(m.group('run_idx'))
        else:
            inferred_model = None
            inferred_run_idx = None
        rows.append({
            'file_path': str(path),
            'file_name': path.name,
            'inferred_model_label': inferred_model,
            'inferred_run_idx': inferred_run_idx,
        })
    return pd.DataFrame(rows)

def read_one_run_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_cols = {'task_id', 'passed'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {sorted(missing)}')

    df = df.copy()
    df['source_file'] = path.name
    df['source_path'] = str(path)

    m = RUN_FILE_PATTERN.match(path.name)
    inferred_model = m.group('model_label') if m else None
    inferred_run_idx = int(m.group('run_idx')) if m else None

    if 'model_label' not in df.columns:
        df['model_label'] = inferred_model
    else:
        df['model_label'] = df['model_label'].fillna(inferred_model)

    if 'run_idx' not in df.columns:
        df['run_idx'] = inferred_run_idx
    else:
        df['run_idx'] = df['run_idx'].fillna(inferred_run_idx)

    df['passed'] = df['passed'].apply(parse_bool_like)
    df['run_idx'] = pd.to_numeric(df['run_idx'], errors='coerce').astype('Int64')

    if 'parse_ok' in df.columns:
        df['parse_ok'] = df['parse_ok'].apply(parse_bool_like)

    keep_cols = [c for c in [
        'task_id', 'completion', 'raw_completion', 'parse_ok', 'model_label',
        'run_idx', 'seed', 'passed', 'result', 'source_file', 'source_path'
    ] if c in df.columns]

    df = df[keep_cols].copy()

    dupes = df.duplicated(subset=['task_id'], keep=False)
    if dupes.any():
        duped_tasks = df.loc[dupes, 'task_id'].tolist()[:10]
        raise ValueError(
            f'{path.name} contains duplicate task_id rows. Example duplicates: {duped_tasks}'
        )

    return df

def load_all_runs(task_results_dir: Path) -> pd.DataFrame:
    files = sorted(task_results_dir.glob('*_task_results.csv'))
    if not files:
        raise FileNotFoundError(f'No *_task_results.csv files found in {task_results_dir}')

    all_parts = []
    for path in files:
        part = read_one_run_csv(path)
        all_parts.append(part)

    all_runs = pd.concat(all_parts, ignore_index=True)

    if all_runs['model_label'].isna().any():
        bad_files = all_runs.loc[all_runs['model_label'].isna(), 'source_file'].unique().tolist()
        raise ValueError(f'Could not infer model_label for files: {bad_files}')

    if all_runs['run_idx'].isna().any():
        bad_files = all_runs.loc[all_runs['run_idx'].isna(), 'source_file'].unique().tolist()
        raise ValueError(f'Could not infer run_idx for files: {bad_files}')

    all_runs['run_idx'] = all_runs['run_idx'].astype(int)
    all_runs['passed_int'] = all_runs['passed'].astype(int)
    return all_runs

def validate_runs(all_runs: pd.DataFrame, expected_runs=10):
    run_counts = (
        all_runs[['model_label', 'run_idx', 'source_file']]
        .drop_duplicates()
        .groupby('model_label')
        .agg(
            n_runs=('run_idx', 'nunique'),
            run_indices=('run_idx', lambda s: sorted(set(s))),
            n_files=('source_file', 'nunique'),
        )
        .reset_index()
        .sort_values('model_label')
    )

    incomplete = run_counts[run_counts['n_runs'] != expected_runs]
    if not incomplete.empty:
        warnings.warn(
            'Some models do not have the expected number of runs.\n'
            + incomplete.to_string(index=False)
        )
    return run_counts

In [ ]:
# =========================
# 5. Helper functions for stability analysis
# =========================
def classify_stability(pass_rate: float, n_runs: int) -> str:
    """Assign an interpretable stability bucket for one model/problem pair."""
    if pd.isna(pass_rate):
        return 'missing'
    if pass_rate == 1.0:
        return 'stable_success'
    if pass_rate == 0.0:
        return 'stable_failure'
    # With 10 runs, 0.8/0.9 or 0.1/0.2 are useful near-stable zones.
    if pass_rate >= 0.8:
        return 'mostly_success'
    if pass_rate <= 0.2:
        return 'mostly_failure'
    return 'unstable_mixed'

def compute_problem_stability(all_runs: pd.DataFrame):
    run_level = all_runs[['model_label', 'run_idx', 'task_id', 'passed_int']].drop_duplicates()

    stability = (
        run_level
        .groupby(['model_label', 'task_id'], as_index=False)
        .agg(
            n_runs=('run_idx', 'nunique'),
            n_passes=('passed_int', 'sum'),
            n_fails=('passed_int', lambda s: int((s == 0).sum())),
            pass_rate=('passed_int', 'mean'),
            pass_sd=('passed_int', 'std'),
        )
    )
    stability['fail_rate'] = 1.0 - stability['pass_rate']
    stability['volatility'] = 4 * stability['pass_rate'] * (1 - stability['pass_rate'])
    stability['stability_bucket'] = stability.apply(
        lambda r: classify_stability(r['pass_rate'], r['n_runs']), axis=1
    )

    # Entropy is another instability measure. 0 = deterministic, 1 = maximally uncertain at pass_rate=0.5.
    p = stability['pass_rate'].clip(1e-12, 1 - 1e-12)
    stability['binary_entropy'] = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

    return run_level, stability

def compute_overall_problem_difficulty(stability: pd.DataFrame):
    overall = (
        stability
        .groupby('task_id', as_index=False)
        .agg(
            mean_pass_rate_across_models=('pass_rate', 'mean'),
            sd_pass_rate_across_models=('pass_rate', 'std'),
            min_pass_rate=('pass_rate', 'min'),
            max_pass_rate=('pass_rate', 'max'),
            mean_volatility=('volatility', 'mean'),
            n_models=('model_label', 'nunique'),
        )
    )
    overall['overall_difficulty'] = 1.0 - overall['mean_pass_rate_across_models']
    overall['between_model_spread'] = overall['max_pass_rate'] - overall['min_pass_rate']
    return overall.sort_values(['overall_difficulty', 'between_model_spread'], ascending=[False, False])

def compute_model_stability_summary(stability: pd.DataFrame):
    summary = (
        stability
        .groupby('model_label', as_index=False)
        .agg(
            mean_problem_pass_rate=('pass_rate', 'mean'),
            median_problem_pass_rate=('pass_rate', 'median'),
            mean_problem_volatility=('volatility', 'mean'),
            median_problem_volatility=('volatility', 'median'),
            n_stable_success=('stability_bucket', lambda s: int((s == 'stable_success').sum())),
            n_mostly_success=('stability_bucket', lambda s: int((s == 'mostly_success').sum())),
            n_unstable_mixed=('stability_bucket', lambda s: int((s == 'unstable_mixed').sum())),
            n_mostly_failure=('stability_bucket', lambda s: int((s == 'mostly_failure').sum())),
            n_stable_failure=('stability_bucket', lambda s: int((s == 'stable_failure').sum())),
            n_tasks=('task_id', 'nunique'),
        )
    )
    return summary.sort_values('mean_problem_pass_rate', ascending=False)

def compute_pairwise_problem_comparison(stability: pd.DataFrame):
    wide = stability.pivot(index='task_id', columns='model_label', values='pass_rate')
    models = list(wide.columns)
    rows = []
    for a in models:
        for b in models:
            if a == b:
                continue
            diff = wide[a] - wide[b]
            rows.append({
                'model_a': a,
                'model_b': b,
                'mean_delta_a_minus_b': diff.mean(),
                'median_delta_a_minus_b': diff.median(),
                'n_tasks_a_better': int((diff > 0).sum()),
                'n_tasks_equal': int((diff == 0).sum()),
                'n_tasks_b_better': int((diff < 0).sum()),
                'largest_gain_task_id': diff.idxmax(),
                'largest_gain': diff.max(),
                'largest_loss_task_id': diff.idxmin(),
                'largest_loss': diff.min(),
            })
    return pd.DataFrame(rows).sort_values(['model_a', 'model_b'])

def find_unique_and_near_unique_solves(stability: pd.DataFrame, threshold=0.8):
    """Find tasks where one model is clearly ahead at the problem level."""
    wide = stability.pivot(index='task_id', columns='model_label', values='pass_rate')
    rows = []
    for task_id, row in wide.iterrows():
        sorted_rates = row.sort_values(ascending=False)
        best_model = sorted_rates.index[0]
        best_rate = sorted_rates.iloc[0]
        second_rate = sorted_rates.iloc[1] if len(sorted_rates) > 1 else np.nan
        margin = best_rate - second_rate if pd.notna(second_rate) else np.nan
        rows.append({
            'task_id': task_id,
            'best_model': best_model,
            'best_pass_rate': best_rate,
            'second_best_pass_rate': second_rate,
            'margin_vs_second_best': margin,
            'unique_stable_or_near_stable_solve': bool(best_rate >= threshold and second_rate < threshold),
        })
    return pd.DataFrame(rows).sort_values(['unique_stable_or_near_stable_solve', 'margin_vs_second_best', 'best_pass_rate'], ascending=[False, False, False])

def order_tasks_for_heatmap(stability: pd.DataFrame):
    overall = compute_overall_problem_difficulty(stability)
    return overall.sort_values(['overall_difficulty', 'between_model_spread'], ascending=[False, False])['task_id'].tolist()

In [ ]:
# =========================
# 6. Discover available files
# =========================
files_df = discover_task_result_files(TASK_RESULTS_DIR)
print(f'Found {len(files_df)} task-result files.')
display(files_df.head(20))

In [ ]:
# =========================
# 7. Load all runs
# =========================
all_runs = load_all_runs(TASK_RESULTS_DIR)

print('Loaded rows:', len(all_runs))
print('Models:', sorted(all_runs['model_label'].unique().tolist()))
display(all_runs.head())

In [ ]:
# =========================
# 8. Validate run and task coverage
# =========================
run_counts = validate_runs(all_runs, expected_runs=EXPECTED_RUNS)
display(run_counts)
run_counts.to_csv(MODEL_RUN_COUNTS_CSV, index=False)

# Check number of tasks per model/run.
task_counts_per_run = (
    all_runs.groupby(['model_label', 'run_idx'], as_index=False)
    .agg(n_tasks=('task_id', 'nunique'))
    .sort_values(['model_label', 'run_idx'])
)
display(task_counts_per_run.head(30))

if task_counts_per_run['n_tasks'].nunique() > 1:
    warnings.warn(
        'Not all runs contain the same number of tasks. Check whether all evaluations completed successfully.'
    )

print('Saved:', MODEL_RUN_COUNTS_CSV)

In [ ]:
# =========================
# 9. Compute problem-level stability tables
# =========================
run_level, problem_stability = compute_problem_stability(all_runs)
problem_overall_difficulty = compute_overall_problem_difficulty(problem_stability)
model_stability_summary = compute_model_stability_summary(problem_stability)
pairwise_problem_comparison = compute_pairwise_problem_comparison(problem_stability)
unique_solves = find_unique_and_near_unique_solves(problem_stability, threshold=0.8)

# Wide table: one row per task, model-specific pass rates and buckets.
pass_rate_wide = problem_stability.pivot(index='task_id', columns='model_label', values='pass_rate')
bucket_wide = problem_stability.pivot(index='task_id', columns='model_label', values='stability_bucket')
problem_stability_wide = pass_rate_wide.add_prefix('pass_rate__').join(bucket_wide.add_prefix('bucket__')).reset_index()
problem_stability_wide = problem_stability_wide.merge(problem_overall_difficulty, on='task_id', how='left')

print('Problem stability by model:')
display(problem_stability.head(20))
print('Overall problem difficulty:')
display(problem_overall_difficulty.head(20))
print('Model-level stability summary:')
display(model_stability_summary)
print('Potential unique / near-unique solves:')
display(unique_solves.head(30))

In [ ]:
# =========================
# 10. Save calculated tables
# =========================
run_level.to_csv(RUN_LEVEL_TASK_MATRIX_CSV, index=False)
problem_stability.to_csv(PROBLEM_STABILITY_CSV, index=False)
problem_stability_wide.to_csv(PROBLEM_STABILITY_WIDE_CSV, index=False)
problem_overall_difficulty.to_csv(PROBLEM_OVERALL_DIFFICULTY_CSV, index=False)
model_stability_summary.to_csv(MODEL_STABILITY_SUMMARY_CSV, index=False)
pairwise_problem_comparison.to_csv(PAIRWISE_PROBLEM_COMPARISON_CSV, index=False)
unique_solves.to_csv(UNIQUE_SOLVES_CSV, index=False)

for path in [
    RUN_LEVEL_TASK_MATRIX_CSV,
    PROBLEM_STABILITY_CSV,
    PROBLEM_STABILITY_WIDE_CSV,
    PROBLEM_OVERALL_DIFFICULTY_CSV,
    MODEL_STABILITY_SUMMARY_CSV,
    PAIRWISE_PROBLEM_COMPARISON_CSV,
    UNIQUE_SOLVES_CSV,
]:
    print('Saved:', path)

# Visualizations

The plots below focus on stability and problem-level behavior rather than aggregate benchmark pass@k.

In [ ]:
# =========================
# 11. Heatmap: per-problem pass rate by model
# =========================
heatmap_df = problem_stability.pivot(index='task_id', columns='model_label', values='pass_rate')
heatmap_df = heatmap_df.loc[order_tasks_for_heatmap(problem_stability)]

fig_height = max(8, 0.07 * len(heatmap_df))
fig, ax = plt.subplots(figsize=(max(8, 1.8 * heatmap_df.shape[1]), fig_height))
im = ax.imshow(heatmap_df.values, aspect='auto', vmin=0, vmax=1)

ax.set_xticks(np.arange(heatmap_df.shape[1]))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(heatmap_df.shape[0]))
ax.set_yticklabels(heatmap_df.index)
ax.set_xlabel('Model')
ax.set_ylabel('HumanEval task_id')
ax.set_title('Problem-level pass rate across runs')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Pass rate across runs')
plt.tight_layout()
plt.savefig(HEATMAP_PNG, dpi=200)
plt.show()

print('Saved:', HEATMAP_PNG)

In [ ]:
# =========================
# 12. Volatility: which tasks are unstable for each model?
# =========================
volatility_df = problem_stability.pivot(index='task_id', columns='model_label', values='volatility')
volatility_df = volatility_df.loc[order_tasks_for_heatmap(problem_stability)]

fig_height = max(8, 0.07 * len(volatility_df))
fig, ax = plt.subplots(figsize=(max(8, 1.8 * volatility_df.shape[1]), fig_height))
im = ax.imshow(volatility_df.values, aspect='auto', vmin=0, vmax=1)

ax.set_xticks(np.arange(volatility_df.shape[1]))
ax.set_xticklabels(volatility_df.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(volatility_df.shape[0]))
ax.set_yticklabels(volatility_df.index)
ax.set_xlabel('Model')
ax.set_ylabel('HumanEval task_id')
ax.set_title('Problem-level volatility across runs')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Volatility = 4p(1-p)')
plt.tight_layout()
plt.savefig(VOLATILITY_PNG, dpi=200)
plt.show()

print('Saved:', VOLATILITY_PNG)

In [ ]:
# =========================
# 13. Distribution of problem pass rates by model
# =========================
models = sorted(problem_stability['model_label'].unique())

plt.figure(figsize=(10, 6))
for model in models:
    values = problem_stability.loc[problem_stability['model_label'] == model, 'pass_rate']
    plt.hist(values, bins=np.linspace(0, 1, 12), alpha=0.35, label=model)

plt.xlabel('Problem pass rate across runs')
plt.ylabel('Number of HumanEval problems')
plt.title('Distribution of problem-level pass rates')
plt.legend(title='Model')
plt.tight_layout()
plt.savefig(DIFFICULTY_PNG, dpi=200)
plt.show()

print('Saved:', DIFFICULTY_PNG)

In [ ]:
# =========================
# 14. Stability bucket counts by model
# =========================
bucket_order = ['stable_success', 'mostly_success', 'unstable_mixed', 'mostly_failure', 'stable_failure']
bucket_counts = (
    problem_stability
    .groupby(['model_label', 'stability_bucket'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=bucket_order, fill_value=0)
)

display(bucket_counts)

ax = bucket_counts.plot(kind='bar', stacked=True, figsize=(11, 6))
ax.set_xlabel('Model')
ax.set_ylabel('Number of HumanEval problems')
ax.set_title('Problem stability buckets by model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(STABILITY_BUCKET_PNG, dpi=200)
plt.show()

print('Saved:', STABILITY_BUCKET_PNG)

In [ ]:
# =========================
# 15. Pairwise model deltas in mean problem pass rate
# =========================
delta_matrix = pairwise_problem_comparison.pivot(
    index='model_a', columns='model_b', values='mean_delta_a_minus_b'
)

fig, ax = plt.subplots(figsize=(max(8, 1.5 * len(delta_matrix.columns)), max(6, 1.2 * len(delta_matrix.index))))
im = ax.imshow(delta_matrix.values, aspect='auto', vmin=-1, vmax=1)

ax.set_xticks(np.arange(delta_matrix.shape[1]))
ax.set_xticklabels(delta_matrix.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(delta_matrix.shape[0]))
ax.set_yticklabels(delta_matrix.index)
ax.set_xlabel('Model B')
ax.set_ylabel('Model A')
ax.set_title('Mean problem pass-rate delta: Model A minus Model B')

for i in range(delta_matrix.shape[0]):
    for j in range(delta_matrix.shape[1]):
        value = delta_matrix.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f'{value:.2f}', ha='center', va='center')

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Mean pass-rate difference')
plt.tight_layout()
plt.savefig(PAIRWISE_DELTA_HEATMAP_PNG, dpi=200)
plt.show()

print('Saved:', PAIRWISE_DELTA_HEATMAP_PNG)

In [ ]:
# =========================
# 16. Inspect the most informative problems
# =========================
# These are problems with high between-model spread and/or high average volatility.
informative_tasks = (
    problem_overall_difficulty
    .sort_values(['between_model_spread', 'mean_volatility', 'overall_difficulty'], ascending=False)
    .head(30)
)

display(informative_tasks)

# Detailed model-level view for those tasks.
informative_detail = problem_stability[
    problem_stability['task_id'].isin(informative_tasks['task_id'])
].sort_values(['task_id', 'model_label'])

display(informative_detail)

In [ ]:
# =========================
# 17. Optional: compare one reference model against others problem-by-problem
# =========================
models = sorted(problem_stability['model_label'].unique())
reference_model = REFERENCE_MODEL_LABEL

if reference_model is None:
    baseline_candidates = [m for m in models if 'baseline' in m.lower()]
    reference_model = baseline_candidates[0] if len(baseline_candidates) == 1 else models[0]

if reference_model not in models:
    raise ValueError(f'reference_model={reference_model!r} not found. Available models: {models}')

pass_rate_wide = problem_stability.pivot(index='task_id', columns='model_label', values='pass_rate')
reference_comparison = pass_rate_wide.subtract(pass_rate_wide[reference_model], axis=0)
reference_comparison = reference_comparison.drop(columns=[reference_model])
reference_comparison = reference_comparison.join(problem_overall_difficulty.set_index('task_id')[['overall_difficulty', 'between_model_spread']])
reference_comparison = reference_comparison.sort_values('between_model_spread', ascending=False)

print('Reference model:', reference_model)
display(reference_comparison.head(40))

## How to interpret the outputs

- `pass_rate`: fraction of runs that passed a given HumanEval problem for a model.
- `volatility = 4p(1-p)`: 0 means deterministic success/failure; 1 means maximally unstable at 50% pass rate.
- `stable_success`: passed in every run.
- `stable_failure`: failed in every run.
- `unstable_mixed`: neither mostly success nor mostly failure; these are the best candidates for qualitative inspection.
- `between_model_spread`: max model pass rate minus min model pass rate for a problem. High values indicate problems that distinguish methodologies.